# 개인 소비 데이터 전처리 및 클러스터 해석
이 노트북은 개인 거래 내역을 정리하고 merchant 분류를 수행한 뒤, 현재 소비 패턴이 8개 클러스터 중 어디에 가까운지 해석합니다.

진행 순서:
1. 원본 거래 CSV 전처리
2. merchant_name 기반 카테고리 분류
3. 모델 입력 집계 생성
4. 현재 클러스터 해석 및 편향 점검

In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
BANK_PATH = Path('bank.csv')
CARD_PATH = Path('card.csv')
OUTPUT_PATH = Path('analysis/personal_transactions.csv')
CARD_LIKE_TYPES = {'체크카드', '카드결제', '신한카드'}


def _find_header_row(path: Path, header_name: str) -> int:
    lines = path.read_text(encoding='utf-8').splitlines()
    for index, line in enumerate(lines):
        if line.startswith(f'{header_name},'):
            return index
    raise ValueError(f'{path} 에서 헤더 {header_name} 를 찾지 못했습니다.')


def load_bank_transactions(path: Path) -> pd.DataFrame:
    header_row = _find_header_row(path, '거래일자')
    bank = pd.read_csv(path, skiprows=header_row, engine='python')
    bank['출금(원)'] = pd.to_numeric(bank['출금(원)'], errors='coerce').fillna(0)
    bank = bank[bank['출금(원)'] > 0].copy()
    bank['transaction_datetime'] = pd.to_datetime(
        bank['거래일자'].astype(str) + ' ' + bank['거래시간'].astype(str),
        errors='coerce',
    )
    bank = bank[bank['transaction_datetime'].notna()].copy()
    bank['payment_method'] = bank['적요'].map(lambda value: '카드' if value in CARD_LIKE_TYPES else '계좌')
    bank['merchant_name'] = bank['내용'].fillna('').astype(str).str.strip()
    bank['transaction_detail'] = bank['적요'].fillna('').astype(str).str.strip()
    bank['amount'] = bank['출금(원)'].astype(int)
    bank['source'] = 'bank'
    bank['source_order'] = range(len(bank))
    return bank[[
        'transaction_datetime',
        'payment_method',
        'amount',
        'merchant_name',
        'transaction_detail',
        'source',
        'source_order',
    ]]


def load_card_transactions(path: Path) -> pd.DataFrame:
    header_row = _find_header_row(path, '거래일')
    card = pd.read_csv(path, skiprows=header_row, engine='python')
    card = card[card['거래일'] != '합계'].copy()
    card['이용금액'] = (
        card['이용금액']
        .astype(str)
        .str.replace(',', '', regex=False)
        .pipe(pd.to_numeric, errors='coerce')
        .fillna(0)
    )
    card = card[card['이용금액'] > 0].copy()
    card['transaction_datetime'] = pd.to_datetime(card['거래일'], format='%Y.%m.%d', errors='coerce')
    card = card[card['transaction_datetime'].notna()].copy()
    card['payment_method'] = '카드'
    card['merchant_name'] = card['가맹점명'].fillna('').astype(str).str.strip()
    card['transaction_detail'] = card['상품구분'].fillna('').astype(str).str.strip()
    card['amount'] = card['이용금액'].astype(int)
    card['source'] = 'card'
    card['source_order'] = range(len(card))
    return card[[
        'transaction_datetime',
        'payment_method',
        'amount',
        'merchant_name',
        'transaction_detail',
        'source',
        'source_order',
    ]]


bank_transactions = load_bank_transactions(BANK_PATH)
card_transactions = load_card_transactions(CARD_PATH)

In [ ]:
transactions = pd.concat([bank_transactions, card_transactions], ignore_index=True)
transactions = transactions.sort_values(
    by=['transaction_datetime', 'source_order'],
    ascending=[True, True],
).reset_index(drop=True)
transactions['cumulative_amount'] = transactions['amount'].cumsum()
transactions = transactions.sort_values(
    by=['transaction_datetime', 'source_order'],
    ascending=[False, False],
).reset_index(drop=True)
transactions.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
transactions.head(20)

In [ ]:
print(f'저장 위치: {OUTPUT_PATH}')
print(f'총 거래 수: {len(transactions)}건')
print(f'총 결제 금액: {transactions["amount"].sum():,}원')
transactions[['transaction_datetime', 'payment_method', 'amount', 'cumulative_amount', 'merchant_name', 'source']].head(10)

## Merchant 분류 준비
전처리된 거래에서 merchant_name 목록을 추출하고, GMS 분류 결과를 저장합니다. 분류 결과는 이후 거래 라벨링과 클러스터 입력 집계에 사용됩니다.

In [ ]:
import json
import os
import re
import shutil
import subprocess
import time

GMS_KEY = os.getenv('GMS_KEY', 'input_here')
GMS_URL = 'https://gms.ssafy.io/gmsapi/api.openai.com/v1/chat/completions'
GMS_MODEL = 'gpt-5.2'
MERCHANT_MAP_PATH = Path('analysis/merchant_category_map.csv')
LABELED_PATH = Path('analysis/personal_transactions_labeled.csv')
TRAINING_INPUT_PATH = Path('analysis/personal_training_input.csv')

TARGET_CATEGORIES = [
    '가전제품', '건강/기호식품', '건강/뷰티/마사지', '경기관람', '공연관람', '교육/학원',
    '교통서비스', '외식', '렌탈서비스', '문화서비스', '방문판매', '방송/미디어', '병원/의료',
    '분식', '사무/교육용품', '서적/도서', '선물/완구', '세탁/가사서비스', '수리서비스', '수의업',
    '숙박', '스포츠/레져용품', '시스템/통신', '악기/공예', '여행/유학대행', '유아용품', '육류/회식',
    '음/식료품소매', '음식배달서비스', '의복/의류', '의약/의료품', '인터넷쇼핑', '인테리어/가정용품',
    '일반스포츠', '자동차/유지비', '자동차학원', '전시장', '제과/제빵/떡/케익', '주점', '취미/오락',
    '커피/음료', '패션잡화', '패스트푸드', '학교', '화장품소매',
]
EXCLUDE_LABEL = '제외'

PROMPT = f"""
너는 merchant_name과 거래 맥락을 보고 소비 카테고리를 분류하는 분류기다.
반드시 아래 카테고리 중 하나 또는 '{EXCLUDE_LABEL}' 만 선택한다.

허용 카테고리:
{', '.join(TARGET_CATEGORIES)}

핵심 원칙:
- '{EXCLUDE_LABEL}' 는 개인 간 송금, 충전, 환불, 수수료, 카드대금 납부, 자동이체, 보험료, 금융성 거래처럼 비소비가 명확할 때만 사용한다.
- 일반 법인명이나 기관명이라도 실제 소비처일 가능성이 있으면 '{EXCLUDE_LABEL}' 대신 가장 그럴듯한 소비 카테고리를 선택한다.
- payment_methods, sources, sample_transaction_details, total_amount, txn_count 는 업종 추정 보조 정보다.
- PX/복지단/편의점/마트/쇼핑몰/온라인 결제대행처럼 소비처로 볼 여지가 있으면 최대한 소비 카테고리로 분류한다.
- 출력은 JSON 객체 하나만 반환한다.
- 형식은 {{"items":[{{"merchant_name":"...","category":"...","reason":"짧은 근거"}}]}} 이다.
- reason은 한국어 1문장 이내.
""".strip()

REVIEW_PROMPT = f"""
너는 이미 '{EXCLUDE_LABEL}' 로 분류된 merchant를 재검토하는 검수기다.
다시 '{EXCLUDE_LABEL}' 를 줄 수 있는 경우는 금융성 거래, 송금, 충전, 환불, 수수료처럼 비소비가 명확할 때뿐이다.
그 외에는 가능한 가장 그럴듯한 소비 카테고리를 선택한다.
출력 형식은 {{"items":[{{"merchant_name":"...","category":"...","reason":"짧은 근거"}}]}} 이다.
""".strip()

FORCED_CATEGORY_OVERRIDES = {
    '국군복지단': ('음/식료품소매', '복지단/PX 성격의 판매처로 보고 식료품·생필품 소매로 보정합니다.'),
    '네이버파이낸셜(주)': ('인터넷쇼핑', '온라인 결제대행사이지만 카드 승인 소비처로 해석해 인터넷쇼핑으로 보정합니다.'),
}

STRICT_EXCLUDE_KEYWORDS = [
    '충전', '수수료', '환불', '자동결제', '자동이체', '카드대금', '알림서비스',
    '보험', '페이', '송금', '이체', '결제대금',
]

PERSON_NAME_PATTERN = re.compile(r'^[가-힣]{2,4}$')
DIGIT_ONLY_PATTERN = re.compile(r'^\d+$')


def _post_gms(payload: dict, gms_key: str) -> dict:
    curl_path = shutil.which('curl')
    if not curl_path:
        raise RuntimeError('curl 명령을 찾지 못했습니다. 시스템에 curl 이 필요합니다.')

    command = [
        curl_path,
        GMS_URL,
        '--http1.1',
        '--silent',
        '--show-error',
        '--retry', '5',
        '--retry-all-errors',
        '--retry-delay', '2',
        '--connect-timeout', '20',
        '--max-time', '120',
        '-H', 'Content-Type: application/json',
        '-H', f'Authorization: Bearer {gms_key}',
        '-d', json.dumps(payload, ensure_ascii=False),
    ]
    completed = subprocess.run(command, capture_output=True, text=True, encoding='utf-8')
    if completed.returncode != 0:
        raise RuntimeError(f'curl 호출 실패: {completed.stderr.strip() or completed.stdout.strip()}')
    return json.loads(completed.stdout)


def _normalize_items(items: list[dict]) -> list[dict]:
    normalized = []
    for item in items:
        category = item.get('category', EXCLUDE_LABEL)
        if category not in TARGET_CATEGORIES and category != EXCLUDE_LABEL:
            category = EXCLUDE_LABEL
        normalized.append({
            'merchant_name': str(item.get('merchant_name', '')).strip(),
            'card_tpbuz_nm_2': category,
            'classification_reason': str(item.get('reason', '')).strip(),
        })
    return normalized


def _load_existing_map(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=['merchant_name', 'card_tpbuz_nm_2', 'classification_reason'])
    existing = pd.read_csv(path)
    expected_cols = ['merchant_name', 'card_tpbuz_nm_2', 'classification_reason']
    for col in expected_cols:
        if col not in existing.columns:
            existing[col] = ''
    return existing[expected_cols].drop_duplicates(subset=['merchant_name'], keep='first')


def build_merchant_profiles(transactions_df: pd.DataFrame) -> dict[str, dict]:
    profiles = {}
    grouped = transactions_df.groupby('merchant_name', dropna=False)
    for merchant_name, group in grouped:
        merchant_name = str(merchant_name).strip()
        if not merchant_name:
            continue
        profiles[merchant_name] = {
            'merchant_name': merchant_name,
            'payment_methods': sorted(group['payment_method'].dropna().astype(str).unique().tolist()),
            'sources': sorted(group['source'].dropna().astype(str).unique().tolist()),
            'sample_transaction_details': group['transaction_detail'].dropna().astype(str).str.strip().loc[lambda s: s.ne('')].drop_duplicates().head(3).tolist(),
            'txn_count': int(len(group)),
            'total_amount': int(group['amount'].sum()),
            'card_txn_count': int((group['payment_method'] == '카드').sum()),
        }
    return profiles


def _rule_based_classify(profile: dict) -> dict | None:
    merchant_name = profile['merchant_name']
    lowered = merchant_name.lower()

    if merchant_name in FORCED_CATEGORY_OVERRIDES:
        category, reason = FORCED_CATEGORY_OVERRIDES[merchant_name]
        return {
            'merchant_name': merchant_name,
            'card_tpbuz_nm_2': category,
            'classification_reason': reason,
        }

    if DIGIT_ONLY_PATTERN.match(merchant_name):
        return {
            'merchant_name': merchant_name,
            'card_tpbuz_nm_2': EXCLUDE_LABEL,
            'classification_reason': '숫자 식별자 형태라 실사용 소비처로 보기 어려워 제외합니다.',
        }

    if PERSON_NAME_PATTERN.match(merchant_name):
        return {
            'merchant_name': merchant_name,
            'card_tpbuz_nm_2': EXCLUDE_LABEL,
            'classification_reason': '개인 이름 형태로 보여 송금·이체 성격으로 제외합니다.',
        }

    strict_text = ' '.join([merchant_name] + profile.get('sample_transaction_details', []))
    if any(keyword in strict_text for keyword in STRICT_EXCLUDE_KEYWORDS):
        return {
            'merchant_name': merchant_name,
            'card_tpbuz_nm_2': EXCLUDE_LABEL,
            'classification_reason': '충전·이체·수수료 등 금융성 키워드가 확인되어 제외합니다.',
        }

    keyword_rules = [
        (['약국'], '의약/의료품', '약국 키워드가 있어 의약품 구매처로 분류합니다.'),
        (['의원', '병원', '내과', '소아과', '치과'], '병원/의료', '의료기관 키워드가 있어 병원/의료로 분류합니다.'),
        (['주유소', 'sk', '석유', '오일'], '자동차/유지비', '주유/연료 관련 상호로 차량유지비로 분류합니다.'),
        (['버스', '지하철', '택시', '철도', 'srt', 'sr', '주차', '파킹'], '교통서비스', '교통/주차 관련 상호로 교통서비스로 분류합니다.'),
        (['스타벅스', '커피', '빽다방', '메가', '벤티', '파스쿠찌', '카페'], '커피/음료', '카페/커피 브랜드 키워드가 있어 커피/음료로 분류합니다.'),
        (['cu', 'gs25', '세븐일레븐', '이마트24', '마트', '편의점'], '음/식료품소매', '편의점·마트 키워드가 있어 식료품 소매로 분류합니다.'),
        (['다이소'], '인테리어/가정용품', '생활용품 매장 키워드가 있어 가정용품으로 분류합니다.'),
        (['맥주', '이자카야', '주점'], '주점', '주류 판매 업장 키워드가 있어 주점으로 분류합니다.'),
        (['삼겹살', '고기', '식육'], '육류/회식', '육류 중심 외식 키워드가 있어 육류/회식으로 분류합니다.'),
        (['떡볶이', '김밥', '우동', '분식'], '분식', '분식류 키워드가 있어 분식으로 분류합니다.'),
        (['버거', '롯데리아', 'bhc'], '패스트푸드', '패스트푸드 브랜드 키워드가 있어 패스트푸드로 분류합니다.'),
        (['쿠팡', '네이버'], '인터넷쇼핑', '온라인 쇼핑/플랫폼 키워드가 있어 인터넷쇼핑으로 분류합니다.'),
    ]

    for keywords, category, reason in keyword_rules:
        if any(keyword in lowered or keyword in merchant_name for keyword in keywords):
            return {
                'merchant_name': merchant_name,
                'card_tpbuz_nm_2': category,
                'classification_reason': reason,
            }

    return None


def _build_batch_payload(profiles: list[dict], prompt: str) -> dict:
    user_items = [
        {
            'merchant_name': profile['merchant_name'],
            'payment_methods': profile['payment_methods'],
            'sources': profile['sources'],
            'sample_transaction_details': profile['sample_transaction_details'],
            'txn_count': profile['txn_count'],
            'total_amount': profile['total_amount'],
        }
        for profile in profiles
    ]
    return {
        'model': GMS_MODEL,
        'messages': [
            {'role': 'developer', 'content': 'Answer in Korean'},
            {'role': 'developer', 'content': prompt},
            {'role': 'user', 'content': json.dumps(user_items, ensure_ascii=False)},
        ],
        'response_format': {'type': 'json_object'},
        'temperature': 0,
    }


def _should_review_exclude(profile: dict, mapped_category: str) -> bool:
    if mapped_category != EXCLUDE_LABEL:
        return False
    merchant_name = profile['merchant_name']
    joined_detail = ' '.join(profile.get('sample_transaction_details', []))
    joined_text = f"{merchant_name} {joined_detail}"

    if DIGIT_ONLY_PATTERN.match(merchant_name) or PERSON_NAME_PATTERN.match(merchant_name):
        return False
    if any(keyword in joined_text for keyword in STRICT_EXCLUDE_KEYWORDS):
        return False
    return profile.get('card_txn_count', 0) > 0


def classify_merchants_via_gms(merchant_names: list[str], merchant_profiles: dict[str, dict], gms_key: str, batch_size: int = 8, sleep_seconds: float = 1.0) -> pd.DataFrame:
    if not gms_key or gms_key == 'input_here':
        raise ValueError('GMS_KEY 환경변수를 설정한 뒤 다시 실행하세요.')

    existing = _load_existing_map(MERCHANT_MAP_PATH)
    existing_records = existing.to_dict('records')
    done_names = set(existing['merchant_name'].astype(str))

    rule_based = []
    for merchant_name in merchant_names:
        if merchant_name in done_names:
            continue
        profile = merchant_profiles[merchant_name]
        classified = _rule_based_classify(profile)
        if classified is not None:
            rule_based.append(classified)

    if rule_based:
        done_names.update(item['merchant_name'] for item in rule_based)
        existing_records.extend(rule_based)

    remaining_profiles = [merchant_profiles[name] for name in merchant_names if name not in done_names]

    print(f'기존 저장 {len(existing)}개, 규칙 분류 {len(rule_based)}개, 신규 GMS 대상 {len(remaining_profiles)}개')
    collected = existing_records

    for start in range(0, len(remaining_profiles), batch_size):
        batch_profiles = remaining_profiles[start:start + batch_size]
        payload = _build_batch_payload(batch_profiles, PROMPT)
        body = _post_gms(payload, gms_key)
        content = body['choices'][0]['message']['content']
        parsed = json.loads(content)
        items = parsed.get('items')
        if not isinstance(items, list):
            raise ValueError(f'응답 파싱 실패: {content}')

        normalized = _normalize_items(items)
        collected.extend(normalized)
        checkpoint = pd.DataFrame(collected).drop_duplicates(subset=['merchant_name'], keep='first')
        checkpoint.to_csv(MERCHANT_MAP_PATH, index=False, encoding='utf-8-sig')
        print(f'{min(start + batch_size, len(remaining_profiles))}/{len(remaining_profiles)} 신규 merchant 분류 완료')
        time.sleep(sleep_seconds)

    result_df = pd.DataFrame(collected).drop_duplicates(subset=['merchant_name'], keep='first')

    review_candidates = [
        merchant_profiles[name]
        for name in merchant_names
        if name in result_df['merchant_name'].values
        and _should_review_exclude(
            merchant_profiles[name],
            result_df.loc[result_df['merchant_name'] == name, 'card_tpbuz_nm_2'].iloc[0],
        )
    ]

    if review_candidates:
        print(f'제외 재검토 대상 {len(review_candidates)}개')
        review_updates = []
        for start in range(0, len(review_candidates), batch_size):
            batch_profiles = review_candidates[start:start + batch_size]
            payload = _build_batch_payload(batch_profiles, REVIEW_PROMPT)
            body = _post_gms(payload, gms_key)
            content = body['choices'][0]['message']['content']
            parsed = json.loads(content)
            items = parsed.get('items')
            if not isinstance(items, list):
                raise ValueError(f'재검토 응답 파싱 실패: {content}')
            review_updates.extend(_normalize_items(items))
            time.sleep(sleep_seconds)

        if review_updates:
            review_df = pd.DataFrame(review_updates).drop_duplicates(subset=['merchant_name'], keep='last')
            result_df = result_df.set_index('merchant_name')
            review_df = review_df.set_index('merchant_name')
            result_df.update(review_df)
            result_df = result_df.reset_index()

    missing = sorted(set(merchant_names) - set(result_df['merchant_name']))
    if missing:
        raise ValueError(f'분류 누락 merchant_name: {missing[:10]}')

    result_df = result_df.sort_values('merchant_name').reset_index(drop=True)
    result_df.to_csv(MERCHANT_MAP_PATH, index=False, encoding='utf-8-sig')
    return result_df

In [ ]:
FORCED_CATEGORY_OVERRIDES.update({
    '국군복지단': ('음/식료품소매', '복지단/PX 성격의 판매처로 보고 식료품·생필품 소매로 보정합니다.'),
    '네이버파이낸셜(주)': ('인터넷쇼핑', '온라인 결제대행사이지만 카드 승인 소비처로 해석해 인터넷쇼핑으로 보정합니다.'),
    '쿠팡': ('인터넷쇼핑', '대표적인 온라인 쇼핑몰 결제이므로 인터넷쇼핑으로 보정합니다.'),
    '꾸에': ('외식', '음식점 상호로 반복 결제되어 외식으로 보정합니다.'),
    '번네식당': ('외식', '식당 상호가 명확해 외식으로 보정합니다.'),
    '마고나비': ('외식', '일반 음식점 상호로 보고 외식으로 보정합니다.'),
    '영우동': ('외식', '우동 전문 음식점 상호로 보여 외식으로 보정합니다.'),
    '차이앤웍': ('외식', '중식 계열 음식점 상호로 보여 외식으로 보정합니다.'),
})


def _has_strict_exclude_signal(profile: dict) -> bool:
    merchant_name = str(profile.get('merchant_name', '') or '')
    details = [str(item) for item in profile.get('sample_transaction_details', []) if str(item)]
    joined_text = ' '.join([merchant_name] + details)
    keywords = [keyword for keyword in STRICT_EXCLUDE_KEYWORDS if isinstance(keyword, str)]
    return any(keyword in joined_text for keyword in keywords)


def _looks_like_person_name_only(profile: dict) -> bool:
    merchant_name = str(profile.get('merchant_name', '') or '').strip()
    if not PERSON_NAME_PATTERN.match(merchant_name):
        return False
    if int(profile.get('card_txn_count', 0)) > 0:
        return False
    payment_methods = set(profile.get('payment_methods', []))
    return payment_methods.issubset({'계좌'})


def _rule_based_classify(profile: dict) -> dict | None:
    merchant_name = str(profile.get('merchant_name', '') or '').strip()
    lowered = merchant_name.lower()

    if merchant_name in FORCED_CATEGORY_OVERRIDES:
        category, reason = FORCED_CATEGORY_OVERRIDES[merchant_name]
        return {
            'merchant_name': merchant_name,
            'card_tpbuz_nm_2': category,
            'classification_reason': reason,
        }

    if DIGIT_ONLY_PATTERN.match(merchant_name):
        return {
            'merchant_name': merchant_name,
            'card_tpbuz_nm_2': EXCLUDE_LABEL,
            'classification_reason': '숫자 식별자 형태라 실사용 소비처로 보기 어려워 제외합니다.',
        }

    if _looks_like_person_name_only(profile):
        return {
            'merchant_name': merchant_name,
            'card_tpbuz_nm_2': EXCLUDE_LABEL,
            'classification_reason': '계좌 기반 개인 이름 거래로 보여 송금·이체 성격으로 제외합니다.',
        }

    if _has_strict_exclude_signal(profile):
        return {
            'merchant_name': merchant_name,
            'card_tpbuz_nm_2': EXCLUDE_LABEL,
            'classification_reason': '충전·이체·수수료 등 금융성 키워드가 확인되어 제외합니다.',
        }

    keyword_rules = [
        (['약국'], '의약/의료품', '약국 키워드가 있어 의약품 구매처로 분류합니다.'),
        (['의원', '병원', '내과', '소아과', '치과'], '병원/의료', '의료기관 키워드가 있어 병원/의료로 분류합니다.'),
        (['주유소', 'sk', '석유', '오일'], '자동차/유지비', '주유/연료 관련 상호로 차량유지비로 분류합니다.'),
        (['버스', '지하철', '택시', '철도', 'srt', 'sr', '주차', '파킹'], '교통서비스', '교통/주차 관련 상호로 교통서비스로 분류합니다.'),
        (['스타벅스', '커피', '빽다방', '메가', '벤티', '파스쿠찌', '카페'], '커피/음료', '카페/커피 브랜드 키워드가 있어 커피/음료로 분류합니다.'),
        (['cu', 'gs25', '세븐일레븐', '이마트24', '마트', '편의점'], '음/식료품소매', '편의점·마트 키워드가 있어 식료품 소매로 분류합니다.'),
        (['다이소'], '인테리어/가정용품', '생활용품 매장 키워드가 있어 가정용품으로 분류합니다.'),
        (['맥주', '이자카야', '주점'], '주점', '주류 판매 업장 키워드가 있어 주점으로 분류합니다.'),
        (['삼겹살', '고기', '식육'], '육류/회식', '육류 중심 외식 키워드가 있어 육류/회식으로 분류합니다.'),
        (['떡볶이', '김밥', '우동', '분식'], '분식', '분식류 키워드가 있어 분식으로 분류합니다.'),
        (['버거', '롯데리아', 'bhc'], '패스트푸드', '패스트푸드 브랜드 키워드가 있어 패스트푸드로 분류합니다.'),
        (['쿠팡', '네이버'], '인터넷쇼핑', '온라인 쇼핑/플랫폼 키워드가 있어 인터넷쇼핑으로 분류합니다.'),
        (['식당', '웍', '라면'], '외식', '음식점 상호 키워드가 있어 외식으로 분류합니다.'),
    ]

    for keywords, category, reason in keyword_rules:
        if any(keyword in lowered or keyword in merchant_name for keyword in keywords):
            return {
                'merchant_name': merchant_name,
                'card_tpbuz_nm_2': category,
                'classification_reason': reason,
            }

    return None


def _should_review_exclude(profile: dict, mapped_category: str) -> bool:
    if mapped_category != EXCLUDE_LABEL:
        return False

    merchant_name = str(profile.get('merchant_name', '') or '')
    if DIGIT_ONLY_PATTERN.match(merchant_name):
        return False
    if _looks_like_person_name_only(profile):
        return False
    if _has_strict_exclude_signal(profile):
        return False
    return int(profile.get('card_txn_count', 0)) > 0

In [ ]:
def _extract_gms_message_content(body: dict) -> str:

    choices = body.get('choices')

    if not isinstance(choices, list) or not choices:

        raise ValueError(f'GMS 응답에 choices 가 없습니다: {body}')

    message = choices[0].get('message', {})

    content = message.get('content')

    if not isinstance(content, str) or not content.strip():

        raise ValueError(f'GMS 응답 content 가 비어 있습니다: {body}')

    return content





def classify_merchants_via_gms(merchant_names: list[str], merchant_profiles: dict[str, dict], gms_key: str, batch_size: int = 8, sleep_seconds: float = 1.0) -> pd.DataFrame:

    if not gms_key or gms_key == 'input_here':

        raise ValueError('GMS_KEY 환경변수를 설정한 뒤 다시 실행하세요.')



    existing = _load_existing_map(MERCHANT_MAP_PATH)

    existing_records = existing.to_dict('records')

    done_names = set(existing['merchant_name'].astype(str))



    rule_based = []

    for merchant_name in merchant_names:

        if merchant_name in done_names:

            continue

        profile = merchant_profiles[merchant_name]

        classified = _rule_based_classify(profile)

        if classified is not None:

            rule_based.append(classified)



    if rule_based:

        done_names.update(item['merchant_name'] for item in rule_based)

        existing_records.extend(rule_based)



    remaining_profiles = [merchant_profiles[name] for name in merchant_names if name not in done_names]



    print(f'기존 저장 {len(existing)}개, 규칙 분류 {len(rule_based)}개, 신규 GMS 대상 {len(remaining_profiles)}개')

    collected = existing_records



    for start in range(0, len(remaining_profiles), batch_size):

        batch_profiles = remaining_profiles[start:start + batch_size]

        payload = _build_batch_payload(batch_profiles, PROMPT)

        body = _post_gms(payload, gms_key)

        content = _extract_gms_message_content(body)

        parsed = json.loads(content)

        items = parsed.get('items')

        if not isinstance(items, list):

            raise ValueError(f'응답 파싱 실패: {content}')



        normalized = _normalize_items(items)

        collected.extend(normalized)

        checkpoint = pd.DataFrame(collected).drop_duplicates(subset=['merchant_name'], keep='first')

        checkpoint.to_csv(MERCHANT_MAP_PATH, index=False, encoding='utf-8-sig')

        print(f'{min(start + batch_size, len(remaining_profiles))}/{len(remaining_profiles)} 신규 merchant 분류 완료')

        time.sleep(sleep_seconds)



    result_df = pd.DataFrame(collected).drop_duplicates(subset=['merchant_name'], keep='first')



    review_candidates = [

        merchant_profiles[name]

        for name in merchant_names

        if name in result_df['merchant_name'].values

        and _should_review_exclude(

            merchant_profiles[name],

            result_df.loc[result_df['merchant_name'] == name, 'card_tpbuz_nm_2'].iloc[0],

        )

    ]



    if review_candidates:

        print(f'제외 재검토 대상 {len(review_candidates)}개')

        review_updates = []

        for start in range(0, len(review_candidates), batch_size):

            batch_profiles = review_candidates[start:start + batch_size]

            payload = _build_batch_payload(batch_profiles, REVIEW_PROMPT)

            try:

                body = _post_gms(payload, gms_key)

                content = _extract_gms_message_content(body)

                parsed = json.loads(content)

                items = parsed.get('items')

                if not isinstance(items, list):

                    raise ValueError(f'재검토 응답 파싱 실패: {content}')

                review_updates.extend(_normalize_items(items))

            except Exception as exc:

                print(f'제외 재검토 배치 건너뜀: {exc}')

            time.sleep(sleep_seconds)



        if review_updates:

            review_df = pd.DataFrame(review_updates).drop_duplicates(subset=['merchant_name'], keep='last')

            result_df = result_df.set_index('merchant_name')

            review_df = review_df.set_index('merchant_name')

            result_df.update(review_df)

            result_df = result_df.reset_index()



    missing = sorted(set(merchant_names) - set(result_df['merchant_name']))

    if missing:

        raise ValueError(f'분류 누락 merchant_name: {missing[:10]}')



    result_df = result_df.sort_values('merchant_name').reset_index(drop=True)

    result_df.to_csv(MERCHANT_MAP_PATH, index=False, encoding='utf-8-sig')

    return result_df


## 사용자 카테고리 재설정
아래 셀은 사용자가 나중에 직접 merchant 카테고리를 덮어쓰는 구간입니다.
기본값은 비워 두고 [analysis/user_category_overrides.csv](analysis/user_category_overrides.csv) 템플릿만 생성합니다.
재설정이 필요할 때만 이 파일에 `merchant_name`, `category`, `reason`을 추가한 뒤 11번째 셀부터 다시 실행하면 됩니다.

In [ ]:
USER_OVERRIDE_PATH = Path('analysis/user_category_overrides.csv')
USER_CATEGORY_OVERRIDE_ROWS: list[dict] = []


def load_user_category_overrides(path: Path, seed_rows: list[dict]) -> pd.DataFrame:
    columns = ['merchant_name', 'category', 'reason']
    if path.exists():
        override_df = pd.read_csv(path)
    else:
        override_df = pd.DataFrame(seed_rows, columns=columns)
        path.parent.mkdir(parents=True, exist_ok=True)
        override_df.to_csv(path, index=False, encoding='utf-8-sig')

    for column in columns:
        if column not in override_df.columns:
            override_df[column] = ''

    override_df = override_df[columns].copy()
    override_df['merchant_name'] = override_df['merchant_name'].fillna('').astype(str).str.strip()
    override_df['category'] = override_df['category'].fillna('').astype(str).str.strip()
    override_df['reason'] = override_df['reason'].fillna('').astype(str).str.strip()
    override_df = override_df.loc[override_df['merchant_name'].ne('')].drop_duplicates(subset=['merchant_name'], keep='last')

    invalid_categories = sorted(
        set(override_df['category']) - set(TARGET_CATEGORIES)
    )
    if invalid_categories:
        raise ValueError(f'허용되지 않은 사용자 카테고리: {invalid_categories}')

    template_df = override_df if not override_df.empty else pd.DataFrame(columns=columns)
    template_df.to_csv(path, index=False, encoding='utf-8-sig')
    return override_df.reset_index(drop=True)


user_override_df = load_user_category_overrides(USER_OVERRIDE_PATH, USER_CATEGORY_OVERRIDE_ROWS)
FORCED_CATEGORY_OVERRIDES.update({
    row.merchant_name: (
        row.category,
        row.reason or '사용자 지정 카테고리 재설정입니다.',
    )
    for row in user_override_df.itertuples(index=False)
})

STRICT_EXCLUDE_KEYWORDS = [keyword for keyword in STRICT_EXCLUDE_KEYWORDS if keyword != '페이']
for keyword in ['이월약정', '충전', '수수료', '환불', '자동결제', '자동이체', '카드대금', '알림서비스', '보험', '송금', '이체', '결제대금']:
    if keyword not in STRICT_EXCLUDE_KEYWORDS:
        STRICT_EXCLUDE_KEYWORDS.append(keyword)

REVIEW_PROMPT = f"""
너는 이미 '{EXCLUDE_LABEL}' 로 분류된 merchant를 재검토하는 검수기다.
다시 '{EXCLUDE_LABEL}' 를 줄 수 있는 경우는 금융성 거래, 송금, 충전, 환불, 수수료처럼 비소비가 명확할 때뿐이다.
- payment_methods 에 '카드'가 있거나 sample_transaction_details 에 '체크카드', '카드결제', '카드승인' 단서가 있으면 실제 소비처일 가능성을 우선 검토한다.
- 한글 2~4자 상호라도 체크카드/카드승인 단서가 있으면 개인송금으로 단정하지 않는다.
- 단, '일부결제금액이월약정', 충전, 송금, 이체, 환불, 수수료, 카드대금, 보험, 자동이체는 금융성 거래로 본다.
- 출력은 반드시 JSON 객체 하나만 반환한다.
출력 형식은 {{"items":[{{"merchant_name":"...","category":"...","reason":"짧은 근거"}}]}} 이다.
""".strip()


def _has_card_spend_signal(profile: dict) -> bool:
    payment_methods = {str(item) for item in profile.get('payment_methods', []) if str(item)}
    details = ' '.join(str(item) for item in profile.get('sample_transaction_details', []) if str(item))
    if '카드' in payment_methods:
        return True
    return any(keyword in details for keyword in ['체크카드', '카드결제', '카드승인'])


def _build_batch_payload(profiles: list[dict], prompt: str) -> dict:
    user_items = [
        {
            'merchant_name': profile['merchant_name'],
            'payment_methods': profile['payment_methods'],
            'sources': profile['sources'],
            'sample_transaction_details': profile['sample_transaction_details'],
            'txn_count': profile['txn_count'],
            'total_amount': profile['total_amount'],
            'card_txn_count': int(profile.get('card_txn_count', 0)),
            'card_spend_signal': _has_card_spend_signal(profile),
        }
        for profile in profiles
    ]
    return {
        'model': GMS_MODEL,
        'messages': [
            {'role': 'developer', 'content': 'Answer in Korean and return JSON only'},
            {'role': 'developer', 'content': prompt},
            {'role': 'user', 'content': json.dumps(user_items, ensure_ascii=False)},
        ],
        'response_format': {'type': 'json_object'},
        'temperature': 0,
    }


def _should_review_exclude(profile: dict, mapped_category: str) -> bool:
    if mapped_category != EXCLUDE_LABEL:
        return False

    merchant_name = str(profile.get('merchant_name', '') or '')
    if DIGIT_ONLY_PATTERN.match(merchant_name):
        return False
    if _has_strict_exclude_signal(profile):
        return False
    if _has_card_spend_signal(profile):
        return True
    if _looks_like_person_name_only(profile):
        return False
    return int(profile.get('card_txn_count', 0)) > 0


print(f'사용자 카테고리 override {len(user_override_df)}개 적용: {USER_OVERRIDE_PATH}')
user_override_df

In [ ]:
merchant_names = sorted(
    transactions['merchant_name']
    .dropna()
    .astype(str)
    .str.strip()
    .loc[lambda series: series.ne('')]
    .unique()
)
merchant_profiles = build_merchant_profiles(transactions)

print(f'분류 대상 merchant_name 수: {len(merchant_names)}개')
print(f'merchant profile 수: {len(merchant_profiles)}개')
merchant_names[:20]

In [ ]:
merchant_category_map = classify_merchants_via_gms(merchant_names, merchant_profiles, GMS_KEY)
merchant_category_map.to_csv(MERCHANT_MAP_PATH, index=False, encoding='utf-8-sig')
print(merchant_category_map['card_tpbuz_nm_2'].value_counts().head(10))
merchant_category_map.head(20)

In [ ]:
if 'merchant_category_map' not in globals():
    merchant_category_map = _load_existing_map(MERCHANT_MAP_PATH)
if 'merchant_profiles' not in globals():
    merchant_profiles = build_merchant_profiles(transactions)

if merchant_category_map.empty:
    raise ValueError('merchant_category_map 이 비어 있습니다. 먼저 7번째 셀에서 merchant 분류를 완료하세요.')

rule_updates = []
for row in merchant_category_map.itertuples(index=False):
    if row.card_tpbuz_nm_2 != EXCLUDE_LABEL:
        continue
    profile = merchant_profiles.get(row.merchant_name)
    if not profile or not _should_review_exclude(profile, row.card_tpbuz_nm_2):
        continue
    refined = _rule_based_classify(profile)
    if refined is not None and refined['card_tpbuz_nm_2'] != EXCLUDE_LABEL:
        rule_updates.append(refined)

if rule_updates:
    rule_update_df = pd.DataFrame(rule_updates).drop_duplicates(subset=['merchant_name'], keep='last')
    merchant_category_map = merchant_category_map.set_index('merchant_name')
    rule_update_df = rule_update_df.set_index('merchant_name')
    merchant_category_map.update(rule_update_df)
    merchant_category_map = merchant_category_map.reset_index()
    merchant_category_map.to_csv(MERCHANT_MAP_PATH, index=False, encoding='utf-8-sig')
    print(f'규칙 보정으로 제외 해제된 merchant 수: {len(rule_update_df)}개')

labeled_transactions = transactions.merge(merchant_category_map, on='merchant_name', how='left')
labeled_transactions['card_tpbuz_nm_2'] = labeled_transactions['card_tpbuz_nm_2'].fillna(EXCLUDE_LABEL)
labeled_transactions.to_csv(LABELED_PATH, index=False, encoding='utf-8-sig')

training_input = (
    labeled_transactions[labeled_transactions['card_tpbuz_nm_2'] != EXCLUDE_LABEL]
    .groupby('card_tpbuz_nm_2', as_index=False)
    .agg(
        amt=('amount', 'sum'),
        cnt=('amount', 'size'),
    )
    .sort_values(['amt', 'cnt'], ascending=[False, False])
    .reset_index(drop=True)
)
training_input.to_csv(TRAINING_INPUT_PATH, index=False, encoding='utf-8-sig')

excluded_breakdown = (
    labeled_transactions[labeled_transactions['card_tpbuz_nm_2'] == EXCLUDE_LABEL]
    .groupby('merchant_name', as_index=False)
    .agg(
        excluded_amt=('amount', 'sum'),
        excluded_cnt=('amount', 'size'),
    )
    .sort_values(['excluded_amt', 'excluded_cnt'], ascending=[False, False])
    .reset_index(drop=True)
)

print(f'라벨 저장: {LABELED_PATH}')
print(f'학습 입력 저장: {TRAINING_INPUT_PATH}')
print(f"제외 금액 합계: {int(excluded_breakdown['excluded_amt'].sum()):,}원")
excluded_breakdown.head(15)

## 클러스터 해석
아래부터는 저장된 모델 결과를 기준으로 현재 소비 패턴을 해석합니다. 고정 설명보다 학습 결과 요약 파일의 시그니처와 상위 카테고리를 우선 사용합니다.

## 한눈에 보기
아래 셀은 현재 사용자 요약, 핵심 카테고리, 후보 클러스터, 제외 비중을 한 번에 볼 수 있도록 정리한 대시보드입니다.

In [ ]:
import importlib
import cluster_definitions
import gmm_predict
from IPython.display import display

cluster_definitions = importlib.reload(cluster_definitions)
gmm_predict = importlib.reload(gmm_predict)
get_cluster_description = cluster_definitions.get_cluster_description
get_cluster_name = cluster_definitions.get_cluster_name
predict_spending_type = gmm_predict.predict_spending_type

if 'training_input' not in globals():
    training_input = pd.read_csv(TRAINING_INPUT_PATH)
if 'labeled_transactions' not in globals():
    labeled_transactions = pd.read_csv(LABELED_PATH)


def build_cluster_feature_rows(cluster_row: pd.Series, limit: int = 5) -> list[dict]:
    feature_rows = []
    for rank in range(1, limit + 1):
        category = cluster_row.get(f'top{rank}_category')
        if pd.isna(category):
            continue
        feature_rows.append({
            'rank': rank,
            'category': category,
            'cluster_share_pct': float(cluster_row.get(f'top{rank}_cluster_pct', 0.0)),
            'overall_share_pct': float(cluster_row.get(f'top{rank}_overall_pct', 0.0)),
            'lift_vs_overall': float(cluster_row.get(f'top{rank}_lift_vs_overall', 0.0)),
        })
    return feature_rows


def build_dynamic_cluster_name(cluster_row: pd.Series) -> str:
    feature_rows = build_cluster_feature_rows(cluster_row)
    distinctive = [
        row['category']
        for row in sorted(feature_rows, key=lambda item: item['lift_vs_overall'], reverse=True)
        if row['lift_vs_overall'] >= 1.2
    ]
    if len(distinctive) < 2:
        distinctive = [row['category'] for row in feature_rows[:2]]
    return ' · '.join(distinctive[:2]) + ' 중심형'


def build_dynamic_cluster_summary(cluster_row: pd.Series) -> str:
    feature_rows = build_cluster_feature_rows(cluster_row)
    summary_features = []
    for row in sorted(feature_rows, key=lambda item: item['lift_vs_overall'], reverse=True)[:3]:
        summary_features.append(
            f"{row['category']} {row['cluster_share_pct']:.1f}% ({row['lift_vs_overall']:.2f}배)"
        )
    return ', '.join(summary_features) + ' 비중이 상대적으로 높은 유형'


cluster_summary_full = pd.read_csv('model/cluster_summary.csv')
cluster_summary_full['fixed_cluster_name'] = cluster_summary_full['cluster_id'].map(
    lambda cluster_id: get_cluster_name(int(cluster_id))
)
cluster_summary_full['fixed_cluster_description'] = cluster_summary_full['cluster_id'].map(
    lambda cluster_id: get_cluster_description(int(cluster_id))
)
cluster_summary_full['dynamic_cluster_name'] = cluster_summary_full.apply(build_dynamic_cluster_name, axis=1)
cluster_summary_full['dynamic_cluster_summary'] = cluster_summary_full.apply(build_dynamic_cluster_summary, axis=1)

cluster_result = predict_spending_type(training_input.copy())
valid_transactions = labeled_transactions[labeled_transactions['card_tpbuz_nm_2'] != EXCLUDE_LABEL].copy()

category_share = training_input.copy()
category_share['share_pct'] = (category_share['amt'] / category_share['amt'].sum() * 100).round(2)
category_share = category_share.sort_values(['amt', 'cnt'], ascending=[False, False]).reset_index(drop=True)
user_share_map = category_share.set_index('card_tpbuz_nm_2')['share_pct'].to_dict()

payment_summary = (
    valid_transactions.groupby('payment_method', as_index=False)
    .agg(
        amt=('amount', 'sum'),
        cnt=('amount', 'size'),
    )
    .sort_values('amt', ascending=False)
    .reset_index(drop=True)
)
payment_summary['share_pct'] = (payment_summary['amt'] / payment_summary['amt'].sum() * 100).round(2)

excluded_summary = pd.DataFrame([
    {
        'excluded_txn_count': int((labeled_transactions['card_tpbuz_nm_2'] == EXCLUDE_LABEL).sum()),
        'excluded_amt': int(labeled_transactions.loc[labeled_transactions['card_tpbuz_nm_2'] == EXCLUDE_LABEL, 'amount'].sum()),
        'excluded_amt_share_pct': round(
            labeled_transactions.loc[labeled_transactions['card_tpbuz_nm_2'] == EXCLUDE_LABEL, 'amount'].sum()
            / labeled_transactions['amount'].sum() * 100,
            2,
        ),
    }
])

probability_table = pd.DataFrame(
    [
        {
            'cluster_id': int(cluster_key.replace('C', '')),
            'probability_pct': float(probability_pct),
        }
        for cluster_key, probability_pct in cluster_result['전체확률'].items()
    ]
).sort_values('probability_pct', ascending=False).reset_index(drop=True)

probability_table = probability_table.merge(
    cluster_summary_full[[
        'cluster_id',
        'fixed_cluster_name',
        'dynamic_cluster_name',
        'cluster_signature',
        'fixed_cluster_description',
        'dynamic_cluster_summary',
        'headline',
        'profile_count',
        'profile_share_pct',
    ]],
    on='cluster_id',
    how='left',
)

top3_candidates = probability_table.head(3).copy()
predicted_cluster_id = int(cluster_result['cluster_id'])
predicted_cluster_row = cluster_summary_full.loc[
    cluster_summary_full['cluster_id'] == predicted_cluster_id
]
if predicted_cluster_row.empty:
    raise ValueError(f'cluster_summary.csv 에 cluster_id={predicted_cluster_id} 정보가 없습니다.')
predicted_cluster_row = predicted_cluster_row.iloc[0]

predicted_cluster_evidence_rows = []
for row in build_cluster_feature_rows(predicted_cluster_row):
    predicted_cluster_evidence_rows.append({
        '순위': row['rank'],
        '카테고리': row['category'],
        '내 소비 비중(%)': round(float(user_share_map.get(row['category'], 0.0)), 2),
        '클러스터 비중(%)': round(row['cluster_share_pct'], 2),
        '전체 평균(%)': round(row['overall_share_pct'], 2),
        '전체 대비 배수': round(row['lift_vs_overall'], 2),
    })
predicted_cluster_evidence = pd.DataFrame(predicted_cluster_evidence_rows)

current_cluster_summary = pd.DataFrame([
    {
        'cluster_id': predicted_cluster_id,
        'dynamic_cluster_name': predicted_cluster_row['dynamic_cluster_name'],
        'dynamic_cluster_summary': predicted_cluster_row['dynamic_cluster_summary'],
        'fixed_cluster_name': predicted_cluster_row['fixed_cluster_name'],
        'fixed_cluster_description': predicted_cluster_row['fixed_cluster_description'],
        'cluster_signature': predicted_cluster_row['cluster_signature'],
        'headline': predicted_cluster_row['headline'],
        'profile_count': int(predicted_cluster_row['profile_count']),
        'profile_share_pct': float(predicted_cluster_row['profile_share_pct']),
        'top1_probability_pct': float(top3_candidates.iloc[0]['probability_pct']),
        'top2_probability_pct': float(top3_candidates.iloc[1]['probability_pct']) if len(top3_candidates) > 1 else 0.0,
    }
])

result_snapshot = pd.DataFrame([
    {
        '현재 클러스터': f"C{predicted_cluster_id}",
        '재해석 라벨': predicted_cluster_row['dynamic_cluster_name'],
        '고정 라벨': predicted_cluster_row['fixed_cluster_name'],
        'Top1 확률(%)': float(top3_candidates.iloc[0]['probability_pct']),
        'Top2 확률(%)': float(top3_candidates.iloc[1]['probability_pct']) if len(top3_candidates) > 1 else 0.0,
        '학습 내 비중(%)': float(predicted_cluster_row['profile_share_pct']),
        '제외 금액 비중(%)': float(excluded_summary.loc[0, 'excluded_amt_share_pct']),
    }
])

category_share_view = category_share.head(10).rename(
    columns={'card_tpbuz_nm_2': '카테고리', 'amt': '금액', 'cnt': '건수', 'share_pct': '비중(%)'}
)
payment_summary_view = payment_summary.rename(
    columns={'payment_method': '결제수단', 'amt': '금액', 'cnt': '건수', 'share_pct': '비중(%)'}
)
top3_candidates_view = top3_candidates[[
    'cluster_id', 'dynamic_cluster_name', 'fixed_cluster_name', 'probability_pct', 'profile_share_pct'
 ]].rename(
    columns={
        'cluster_id': '클러스터',
        'dynamic_cluster_name': '재해석 라벨',
        'fixed_cluster_name': '고정 라벨',
        'probability_pct': '확률(%)',
        'profile_share_pct': '학습 비중(%)',
    }
)
excluded_top_view = excluded_breakdown.head(10).rename(
    columns={'merchant_name': 'merchant_name', 'excluded_amt': '제외 금액', 'excluded_cnt': '제외 건수'}
)

print(f"현재 클러스터: C{predicted_cluster_id} | {predicted_cluster_row['dynamic_cluster_name']}")
print(f"재해석 요약: {predicted_cluster_row['dynamic_cluster_summary']}")
print(f"현재 고정 라벨: {predicted_cluster_row['fixed_cluster_name']}")
print(f"해석 기준: {predicted_cluster_row['headline']}")

display(result_snapshot)
display(category_share_view)
display(predicted_cluster_evidence)
display(top3_candidates_view)
display(payment_summary_view)
display(excluded_top_view)

analysis_tables = {
    'result_snapshot': result_snapshot,
    'current_cluster_summary': current_cluster_summary,
    'predicted_cluster_evidence': predicted_cluster_evidence,
    'top3_candidates': top3_candidates_view,
    'category_share_top10': category_share_view,
    'payment_summary': payment_summary_view,
    'excluded_summary': excluded_summary,
    'excluded_top10': excluded_top_view,
}

analysis_tables

In [ ]:
cluster_catalog_rows = []
for row in cluster_summary_full.sort_values('cluster_id').itertuples(index=False):
    top_categories = []
    overlap_categories = []
    overlap_sum = 0.0
    for feature in build_cluster_feature_rows(pd.Series(row._asdict())):
        user_pct = round(float(user_share_map.get(feature['category'], 0.0)), 2)
        top_categories.append(f"{feature['category']}({feature['cluster_share_pct']:.1f}%, {feature['lift_vs_overall']:.2f}배)")
        overlap_categories.append(f"{feature['category']}:{user_pct:.1f}%")
        overlap_sum += user_pct

    current_probability = probability_table.loc[
        probability_table['cluster_id'] == row.cluster_id, 'probability_pct'
    ]
    current_probability = float(current_probability.iloc[0]) if not current_probability.empty else 0.0

    cluster_catalog_rows.append({
        'cluster_id': int(row.cluster_id),
        'dynamic_cluster_name': row.dynamic_cluster_name,
        'fixed_cluster_name': row.fixed_cluster_name,
        'dynamic_cluster_summary': row.dynamic_cluster_summary,
        'profile_share_pct': float(row.profile_share_pct),
        'current_user_probability_pct': round(current_probability, 2),
        'user_overlap_top5_pct': round(overlap_sum, 2),
        'top_categories': ' / '.join(top_categories),
        'user_overlap_by_category': ' / '.join(overlap_categories),
    })

cluster_catalog = pd.DataFrame(cluster_catalog_rows).sort_values(
    ['current_user_probability_pct', 'profile_share_pct'],
    ascending=[False, False],
).reset_index(drop=True)

print('8개 클러스터 재해석 표')
print('동적 라벨과 현재 고정 라벨을 함께 봅니다.')

cluster_catalog

In [ ]:
top2_clusters = top3_candidates.head(2).copy()
compare_rows = []
for cluster_id in top2_clusters['cluster_id']:
    cluster_row = cluster_summary_full.loc[cluster_summary_full['cluster_id'] == cluster_id].iloc[0]
    for feature in build_cluster_feature_rows(cluster_row):
        compare_rows.append({
            'cluster_id': int(cluster_id),
            'dynamic_cluster_name': cluster_row['dynamic_cluster_name'],
            'fixed_cluster_name': cluster_row['fixed_cluster_name'],
            'rank': feature['rank'],
            'category': feature['category'],
            'user_share_pct': round(float(user_share_map.get(feature['category'], 0.0)), 2),
            'cluster_share_pct': round(feature['cluster_share_pct'], 2),
            'overall_share_pct': round(feature['overall_share_pct'], 2),
            'gap_user_vs_cluster_pctp': round(
                float(user_share_map.get(feature['category'], 0.0)) - feature['cluster_share_pct'],
                2,
            ),
        })

top2_cluster_compare = pd.DataFrame(compare_rows)

print('상위 2개 후보 비교')
for row in top2_clusters.itertuples(index=False):
    print(f"- C{row.cluster_id} | {row.dynamic_cluster_name} | 확률 {row.probability_pct}%")
    print(f"  현재 고정 라벨: {row.fixed_cluster_name}")
    print(f"  재해석 요약: {row.dynamic_cluster_summary}")
    print(f"  참고 시그니처: {row.cluster_signature}")

{
    'top2_clusters': top2_clusters,
    'top2_cluster_compare': top2_cluster_compare,
}

In [ ]:
cluster_training_summary = cluster_summary_full[
    ['cluster_id', 'cluster_name', 'profile_count', 'profile_share_pct']
] .sort_values('profile_share_pct', ascending=False).reset_index(drop=True)

training_profile_count = int(cluster_training_summary['profile_count'].sum())
top2_training_share_pct = round(cluster_training_summary.head(2)['profile_share_pct'].sum(), 2)
singleton_cluster_count = int((cluster_training_summary['profile_count'] == 1).sum())
max_min_profile_ratio = round(
    cluster_training_summary['profile_count'].max() / cluster_training_summary['profile_count'].replace(0, pd.NA).min(),
    2,
)

top1_prob = float(top3_candidates.iloc[0]['probability_pct'])
top2_prob = float(top3_candidates.iloc[1]['probability_pct']) if len(top3_candidates) > 1 else 0.0
top3_prob = float(top3_candidates.iloc[2]['probability_pct']) if len(top3_candidates) > 2 else 0.0
prob_gap_top1_top2 = round(top1_prob - top2_prob, 2)
prob_gap_top2_top3 = round(top2_prob - top3_prob, 2)

if top1_prob >= 80 and prob_gap_top1_top2 >= 30:
    raw_confidence_label = '높음'
elif top1_prob >= 55 and prob_gap_top1_top2 >= 15:
    raw_confidence_label = '보통'
else:
    raw_confidence_label = '낮음'

warning_reasons = []
if excluded_summary.loc[0, 'excluded_amt_share_pct'] >= 50:
    warning_reasons.append('제외 금액 비중이 높아 입력 정보 손실이 큼')
if top2_training_share_pct >= 60:
    warning_reasons.append('학습 클러스터 분포가 상위 소수 클러스터에 치우침')
if singleton_cluster_count >= 3:
    warning_reasons.append('단일 프로파일 클러스터가 많아 확률 과신 가능성 있음')
if top2_prob == 0:
    warning_reasons.append('후보 확률이 극단적으로 벌어져 calibrated probability 로 보기 어려움')

if len(warning_reasons) >= 3:
    adjusted_confidence_label = '주의 필요'
elif len(warning_reasons) >= 1 and raw_confidence_label == '높음':
    adjusted_confidence_label = '보통'
else:
    adjusted_confidence_label = raw_confidence_label

bias_and_confidence = pd.DataFrame([
    {
        'training_profile_count': training_profile_count,
        'top2_cluster_share_pct': top2_training_share_pct,
        'singleton_cluster_count': singleton_cluster_count,
        'max_min_profile_ratio': max_min_profile_ratio,
        'excluded_amt_share_pct': excluded_summary.loc[0, 'excluded_amt_share_pct'],
        'top1_probability_pct': top1_prob,
        'top2_probability_pct': top2_prob,
        'gap_top1_top2_pctp': prob_gap_top1_top2,
        'gap_top2_top3_pctp': prob_gap_top2_top3,
        'raw_confidence_label': raw_confidence_label,
        'adjusted_confidence_label': adjusted_confidence_label,
        'warning_count': len(warning_reasons),
    }
])

print('편향 및 신뢰도 점검')
print(f'- 학습 프로파일 수: {training_profile_count}개')
print(f'- 상위 2개 클러스터 비중: {top2_training_share_pct}%')
print(f'- 단일 프로파일 클러스터 수: {singleton_cluster_count}개')
print(f'- 제외 금액 비중: {excluded_summary.loc[0, "excluded_amt_share_pct"]}%')
print(f'- Top1-Top2 격차: {prob_gap_top1_top2}%p')
print(f'- 보정 후 신뢰도: {adjusted_confidence_label}')
if warning_reasons:
    print('- 주의 신호:')
    for reason in warning_reasons:
        print(f'  * {reason}')

{
    'cluster_training_summary': cluster_training_summary,
    'bias_and_confidence': bias_and_confidence,
    'warning_reasons': pd.DataFrame({'warning_reason': warning_reasons}) if warning_reasons else pd.DataFrame(columns=['warning_reason']),
}